# 🚀 Irrigation Need Prediction — Fast SVM (Google Colab + T4 GPU)

**Task**: Predict `Irrigation_Need` (Low / Medium / High) using SVM.

**Speed optimizations**:
- `Nystroem` kernel approximation → transforms RBF kernel into a linear problem → trains in **minutes** instead of hours
- Feature selection with `mutual_info_classif` → keeps only the most discriminative features
- `SGDClassifier(loss='hinge')` → online SVM solver, handles 630K rows efficiently
- All preprocessing in a single leak-free `Pipeline`

---
## 1. Colab Setup

In [ ]:
# --- Check runtime ---
import subprocess
try:
    gpu_info = subprocess.check_output('nvidia-smi', shell=True).decode()
    if 'T4' in gpu_info:
        print('✅ T4 GPU detected')
    else:
        print('⚠ GPU detected but not T4:')
    print(gpu_info.split('\n')[8])  # GPU name line
except:
    print('⚠ No GPU found — running on CPU (will still work, just slower)')

import psutil
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f'RAM: {ram_gb:.1f} GB')

In [ ]:
# --- Mount Google Drive (upload your train.csv and test.csv there) ---
from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# ⚠ SET YOUR DATA PATH HERE
# Put train.csv and test.csv in this folder on your Google Drive
# ============================================================
DATA_DIR = '/content/drive/MyDrive/ML1-2/'

import os
print('Files in DATA_DIR:')
for f in os.listdir(DATA_DIR):
    size_mb = os.path.getsize(os.path.join(DATA_DIR, f)) / (1024**2)
    print(f'  {f} ({size_mb:.1f} MB)')

---
## 2. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import (
    LabelEncoder, StandardScaler, OneHotEncoder
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, GridSearchCV,
    StratifiedShuffleSplit
)
from sklearn.svm import LinearSVC, SVC
from sklearn.linear_model import SGDClassifier
from sklearn.kernel_approximation import Nystroem
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_selection import mutual_info_classif, SelectKBest
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, f1_score
)

SEED = 42
np.random.seed(SEED)

print('All imports ready. ✅')

---
## 3. Load Data

In [ ]:
t0 = time.time()

train_raw = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_raw  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f'Train shape : {train_raw.shape}')
print(f'Test shape  : {test_raw.shape}')
print(f'Loaded in {time.time()-t0:.1f}s')
print()

TARGET = 'Irrigation_Need'
print('Target distribution:')
counts = train_raw[TARGET].value_counts()
for cls, cnt in counts.items():
    print(f'  {cls:8s}: {cnt:>7,} ({cnt/len(train_raw)*100:.1f}%)')
print()
print('Missing values:', train_raw.isnull().sum().sum())

---
## 4. Feature Engineering

Create domain-relevant interaction features that help SVM find better decision boundaries.

In [ ]:
def engineer_features(df):
    """Create domain-relevant interaction and ratio features."""
    df = df.copy()
    eps = 1e-6

    # Interaction features
    df['Moisture_x_Temp']     = df['Soil_Moisture'] * df['Temperature_C']
    df['Rainfall_x_Humidity'] = df['Rainfall_mm'] * df['Humidity']
    df['Temp_x_Humidity']     = df['Temperature_C'] * df['Humidity']
    df['Moisture_x_Rainfall'] = df['Soil_Moisture'] * df['Rainfall_mm']

    # Ratio features
    df['Irrigation_per_Hectare'] = df['Previous_Irrigation_mm'] / (df['Field_Area_hectare'] + eps)
    df['Rainfall_per_Sunlight']  = df['Rainfall_mm'] / (df['Sunlight_Hours'] + eps)
    df['Moisture_Deficit']       = df['Temperature_C'] - df['Soil_Moisture']

    return df

t0 = time.time()
train_fe = engineer_features(train_raw)
test_fe  = engineer_features(test_raw)
print(f'Feature engineering done in {time.time()-t0:.1f}s')
print(f'Train: {train_fe.shape}, Test: {test_fe.shape}')

---
## 5. Feature Selection with Mutual Information

We rank all features by their mutual information with the target and keep the top ones.  
Fewer features = faster SVM training + less noise.

In [ ]:
cat_cols = [
    'Soil_Type', 'Crop_Type', 'Crop_Growth_Stage',
    'Season', 'Irrigation_Type', 'Water_Source',
    'Mulching_Used', 'Region'
]

num_cols = [
    'Soil_pH', 'Soil_Moisture', 'Organic_Carbon',
    'Electrical_Conductivity', 'Temperature_C', 'Humidity',
    'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh',
    'Field_Area_hectare', 'Previous_Irrigation_mm',
    # Engineered
    'Moisture_x_Temp', 'Rainfall_x_Humidity',
    'Temp_x_Humidity', 'Moisture_x_Rainfall',
    'Irrigation_per_Hectare', 'Rainfall_per_Sunlight',
    'Moisture_Deficit'
]

# Encode target
le_target = LabelEncoder()
y_all = le_target.fit_transform(train_fe[TARGET])
print('Target classes:', le_target.classes_)

# --- Compute mutual information on a subsample (fast) ---
MI_SAMPLE = 50_000
rng = np.random.default_rng(SEED)
mi_idx = rng.choice(len(train_fe), size=MI_SAMPLE, replace=False)

# For MI, we need numeric representation — quick label encode categoricals
mi_df = train_fe.iloc[mi_idx].copy()
for col in cat_cols:
    mi_df[col] = LabelEncoder().fit_transform(mi_df[col])

all_feature_cols = num_cols + cat_cols

print(f'\nComputing mutual information on {MI_SAMPLE:,} samples...')
t0 = time.time()
mi_scores = mutual_info_classif(
    mi_df[all_feature_cols], y_all[mi_idx],
    discrete_features=[False]*len(num_cols) + [True]*len(cat_cols),
    random_state=SEED, n_neighbors=5
)
print(f'Done in {time.time()-t0:.1f}s')

# Rank features
mi_ranking = pd.DataFrame({
    'feature': all_feature_cols,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False)

print('\nFeature ranking by Mutual Information:')
print(mi_ranking.to_string(index=False))

# Visualize
plt.figure(figsize=(12, 5))
plt.barh(mi_ranking['feature'][::-1], mi_ranking['mi_score'][::-1], color='steelblue')
plt.xlabel('Mutual Information Score')
plt.title('Feature Importance (Mutual Information with Target)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Select top features ---
# Keep features with MI score above a threshold (drop near-zero features)
MI_THRESHOLD = 0.005  # drop features with negligible information

selected = mi_ranking[mi_ranking['mi_score'] >= MI_THRESHOLD]['feature'].tolist()

selected_num = [f for f in selected if f in num_cols]
selected_cat = [f for f in selected if f in cat_cols]

print(f'Selected {len(selected)} / {len(all_feature_cols)} features (MI >= {MI_THRESHOLD})')
print(f'  Numeric ({len(selected_num)}): {selected_num}')
print(f'  Categorical ({len(selected_cat)}): {selected_cat}')

dropped = mi_ranking[mi_ranking['mi_score'] < MI_THRESHOLD]['feature'].tolist()
if dropped:
    print(f'\n  Dropped ({len(dropped)}): {dropped}')
else:
    print('\n  No features dropped — all are informative')

---
## 6. Preprocessing Pipeline

- `OneHotEncoder` for categoricals (SVM is distance-based — LabelEncoder creates false ordinal relationships)
- `StandardScaler` for numerics (SVM is sensitive to feature magnitudes)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), selected_num),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), selected_cat)
    ],
    remainder='drop'
)

feature_cols = selected_num + selected_cat

X_all_df  = train_fe[feature_cols]
X_test_df = test_fe[feature_cols]
test_ids  = test_fe['id'].values

print(f'Feature matrix: {X_all_df.shape}')
print(f'Test matrix   : {X_test_df.shape}')

---
## 7. Train / Validation Split

In [ ]:
X_train_df, X_val_df, y_train, y_val = train_test_split(
    X_all_df, y_all,
    test_size=0.15,
    random_state=SEED,
    stratify=y_all
)

print(f'Train : {X_train_df.shape[0]:,}')
print(f'Val   : {X_val_df.shape[0]:,}')

---
## 8. Fast SVM — Nystroem Kernel Approximation + SGD

**Why this approach?**

Standard `SVC(kernel='rbf')` has **O(n²) to O(n³)** complexity — training on 630K rows would take **hours**.

Instead, we use a two-step approximation that gives RBF-quality boundaries in **minutes**:
1. **`Nystroem`**: Approximates the RBF kernel map using a random subsample of landmark points. Transforms the input into a feature space where linear separation ≈ RBF separation.
2. **`SGDClassifier(loss='hinge')`**: This IS a linear SVM trained with stochastic gradient descent — handles 630K rows in seconds.

The combination `Nystroem + SGD(hinge)` ≈ RBF SVM, but trains **100x faster**.

In [ ]:
# --- 8a. Quick baseline: LinearSVC (fastest possible SVM) ---
print('='*55)
print('BASELINE: LinearSVC (no kernel approximation)')
print('='*55)

linear_svm_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('svm', LinearSVC(
        C=1.0,
        class_weight='balanced',
        max_iter=5000,
        random_state=SEED,
        dual='auto'
    ))
])

t0 = time.time()
linear_svm_pipe.fit(X_train_df, y_train)
linear_time = time.time() - t0

linear_preds = linear_svm_pipe.predict(X_val_df)
linear_acc = accuracy_score(y_val, linear_preds)
linear_f1  = f1_score(y_val, linear_preds, average='macro')

print(f'Training time   : {linear_time:.1f}s')
print(f'Val Accuracy    : {linear_acc:.4f}')
print(f'Val F1 Macro    : {linear_f1:.4f}')
print()
print(classification_report(y_val, linear_preds, target_names=le_target.classes_, digits=4))

In [ ]:
# --- 8b. Nystroem RBF Approximation + SGD SVM ---
print('='*55)
print('MAIN MODEL: Nystroem RBF + SGDClassifier (hinge loss)')
print('='*55)

# Nystroem n_components: more = better approximation but slower
# 500-1000 is a good trade-off for 630K rows
N_COMPONENTS = 800

nystroem_svm_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('kernel_approx', Nystroem(
        kernel='rbf',
        gamma='scale',
        n_components=N_COMPONENTS,
        random_state=SEED
    )),
    ('svm', SGDClassifier(
        loss='hinge',              # This makes it an SVM (hinge loss = SVM)
        alpha=1e-4,                # Regularization (≈ 1/C)
        class_weight='balanced',   # Handle class imbalance
        max_iter=1000,
        tol=1e-3,
        random_state=SEED,
        n_jobs=-1
    ))
])

t0 = time.time()
nystroem_svm_pipe.fit(X_train_df, y_train)
nystroem_time = time.time() - t0

nystroem_preds = nystroem_svm_pipe.predict(X_val_df)
nystroem_acc = accuracy_score(y_val, nystroem_preds)
nystroem_f1  = f1_score(y_val, nystroem_preds, average='macro')

print(f'Training time   : {nystroem_time:.1f}s')
print(f'Val Accuracy    : {nystroem_acc:.4f}')
print(f'Val F1 Macro    : {nystroem_f1:.4f}')
print()
print(classification_report(y_val, nystroem_preds, target_names=le_target.classes_, digits=4))

---
## 9. Hyperparameter Tuning (Fast GridSearchCV)

We tune `alpha` (regularization, equivalent to 1/C) and `n_components` (kernel approximation quality).

In [ ]:
tuning_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('kernel_approx', Nystroem(
        kernel='rbf',
        gamma='scale',
        random_state=SEED
    )),
    ('svm', SGDClassifier(
        loss='hinge',
        class_weight='balanced',
        max_iter=1000,
        tol=1e-3,
        random_state=SEED,
        n_jobs=-1
    ))
])

param_grid = {
    'kernel_approx__n_components': [500, 800, 1200],
    'kernel_approx__gamma':        ['scale', 0.01, 0.1],
    'svm__alpha':                  [1e-5, 1e-4, 1e-3]
}

n_combos = 3 * 3 * 3
cv_tune = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

print(f'GridSearchCV: {n_combos} combinations × 3 folds = {n_combos * 3} fits')
print(f'Training on {len(X_train_df):,} samples...')
print('Estimated time: ~5-15 minutes on T4\n')

t0 = time.time()
grid_search = GridSearchCV(
    tuning_pipe,
    param_grid,
    cv=cv_tune,
    scoring='f1_macro',
    refit=True,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid_search.fit(X_train_df, y_train)
tune_time = time.time() - t0

print(f'\n{"="*55}')
print(f'Best Parameters  : {grid_search.best_params_}')
print(f'Best F1 Macro    : {grid_search.best_score_:.4f}')
print(f'Total tuning time: {tune_time:.0f}s ({tune_time/60:.1f} min)')
print(f'{"="*55}')

In [ ]:
# --- Display GridSearch results ---
cv_df = pd.DataFrame(grid_search.cv_results_)
cv_display = cv_df[[
    'param_kernel_approx__n_components',
    'param_kernel_approx__gamma',
    'param_svm__alpha',
    'mean_test_score', 'std_test_score',
    'mean_train_score', 'rank_test_score'
]].sort_values('rank_test_score').head(10)

cv_display.columns = ['n_components', 'gamma', 'alpha', 'F1 (test)', 'Std', 'F1 (train)', 'Rank']
print('Top 10 Grid Search Results:')
print(cv_display.to_string(index=False))

---
## 10. Evaluate Best Model on Validation Set

In [ ]:
best_model = grid_search.best_estimator_

val_preds = best_model.predict(X_val_df)
val_acc = accuracy_score(y_val, val_preds)
val_f1  = f1_score(y_val, val_preds, average='macro')

print('='*55)
print('BEST SVM — VALIDATION RESULTS')
print('='*55)
print(f'Accuracy  : {val_acc:.4f}')
print(f'F1 Macro  : {val_f1:.4f}')
print()
print(classification_report(
    y_val, val_preds,
    target_names=le_target.classes_,
    digits=4
))

In [ ]:
# --- Confusion Matrix ---
cm = confusion_matrix(y_val, val_preds)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_target.classes_,
            yticklabels=le_target.classes_, ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')

cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=le_target.classes_,
            yticklabels=le_target.classes_, ax=axes[1])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix (Normalized)', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# --- Model Comparison Summary ---
print('\n' + '='*55)
print('MODEL COMPARISON')
print('='*55)

comparison = pd.DataFrame({
    'Model': ['LinearSVC', 'Nystroem + SGD (default)', 'Nystroem + SGD (tuned)'],
    'Accuracy': [linear_acc, nystroem_acc, val_acc],
    'F1 Macro': [linear_f1, nystroem_f1, val_f1],
    'Train Time (s)': [linear_time, nystroem_time, tune_time]
})
print(comparison.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(comparison))
width = 0.35
bars1 = ax.bar(x - width/2, comparison['Accuracy'], width, label='Accuracy', color='#2196F3')
bars2 = ax.bar(x + width/2, comparison['F1 Macro'], width, label='F1 Macro', color='#FF9800')
ax.set_xticks(x)
ax.set_xticklabels(comparison['Model'], fontsize=10)
ax.set_ylabel('Score')
ax.set_title('SVM Variants — Performance Comparison', fontweight='bold')
ax.legend()
for b in bars1:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.003, f'{b.get_height():.4f}',
            ha='center', fontsize=9, fontweight='bold')
for b in bars2:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.003, f'{b.get_height():.4f}',
            ha='center', fontsize=9, fontweight='bold')
ax.set_ylim(0, 1.08)
plt.tight_layout()
plt.show()

---
## 11. Retrain on Full Data & Generate Submission

In [ ]:
# Extract best hyperparameters
best_params = grid_search.best_params_
print('Retraining on ALL training data with best params:')
print(f'  n_components = {best_params["kernel_approx__n_components"]}')
print(f'  gamma        = {best_params["kernel_approx__gamma"]}')
print(f'  alpha        = {best_params["svm__alpha"]}')

final_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('kernel_approx', Nystroem(
        kernel='rbf',
        gamma=best_params['kernel_approx__gamma'],
        n_components=best_params['kernel_approx__n_components'],
        random_state=SEED
    )),
    ('svm', SGDClassifier(
        loss='hinge',
        alpha=best_params['svm__alpha'],
        class_weight='balanced',
        max_iter=1000,
        tol=1e-3,
        random_state=SEED,
        n_jobs=-1
    ))
])

# Train on ALL 630K rows
t0 = time.time()
final_pipe.fit(X_all_df, y_all)
print(f'\nFull retrain done in {time.time()-t0:.1f}s')

# Predict test set
print('Predicting test set...')
t0 = time.time()
test_preds_encoded = final_pipe.predict(X_test_df)
test_preds_labels  = le_target.inverse_transform(test_preds_encoded)
print(f'Prediction done in {time.time()-t0:.1f}s')

# Create submission
submission = pd.DataFrame({
    'id': test_ids,
    'Irrigation_Need': test_preds_labels
})

# Save to Drive so you don't lose it
submission_path = os.path.join(DATA_DIR, 'submission.csv')
submission.to_csv(submission_path, index=False)
# Also save to Colab local for quick download
submission.to_csv('submission.csv', index=False)

print(f'\n✅ Submission saved!')
print(f'   Drive : {submission_path}')
print(f'   Local : /content/submission.csv')
print(f'   Shape : {submission.shape}')
print()
print('Prediction distribution:')
print(submission['Irrigation_Need'].value_counts())
print()
submission.head(10)

In [ ]:
# --- Quick download from Colab ---
from google.colab import files
files.download('submission.csv')
print('Download started! Check your browser downloads.')

---
## Summary

| Step | Technique | Why |
|------|-----------|-----|
| **Encoding** | `OneHotEncoder` via `ColumnTransformer` | SVM is distance-based — LabelEncoder corrupts distances |
| **Scaling** | `StandardScaler` | SVM is sensitive to feature magnitudes |
| **Feature Selection** | `mutual_info_classif` | Remove noisy features → faster + better generalization |
| **Feature Engineering** | Interaction & ratio features | Capture nonlinear domain relationships |
| **Class Imbalance** | `class_weight='balanced'` | High class is only ~3.4% of data |
| **Kernel Trick** | `Nystroem` RBF approximation | RBF-quality boundaries in linear time |
| **SVM Solver** | `SGDClassifier(loss='hinge')` | Online SVM — handles 630K rows in seconds |
| **Tuning** | `GridSearchCV` (3-fold, f1_macro) | Systematic hyperparameter optimization |
| **Evaluation** | Classification report + confusion matrix | Per-class precision/recall/F1 |